# P3: Acquisition & Integration — DSA 405 (Fall 2026)
**Unity ID:** nmbrockm  
**Research question:** Do S&P 500 companies with larger year-over-year increases in 10-K Item 1A risk-factor length experience worse subsequent 12-month stock returns?

**Scope of this reproducible milestone:** Acquire the Wikipedia S&P 500 constituent list and SEC EDGAR company submission metadata; join on CIK; verify that join. This is a *company-level source-integration check*, **not** a completed 10-K text / stock-return analysis. Stooq is deliberately excluded until automated-download permission is confirmed. Run all cells with internet access and inspect outputs before submitting.

## 1. Acquisition approach
Wikipedia's public constituent table supplies ticker, company and CIK. SEC's public JSON submission endpoint supplies company name, CIK and recent filing metadata. This notebook caches exact raw HTTP response bytes in `data/raw/`, enforces a minimum one-second gap between requests, and stops on malformed responses, unexpected page structures, HTTP errors and missing 10-K metadata. It uses no login or bypass. **The API response-header rate-limit check below examines headers actually returned, rather than assuming a limit from documentation.** If no rate-limit header is provided, it says so and still applies the conservative 1-second interval.

For a bounded, easily reproducible demonstration, use the first **5** Wikipedia constituents sorted by ticker. Change `N_COMPANIES` to increase coverage after verifying the source's access terms and acceptable load. No pagination applies to Wikipedia's single constituent table; the SEC endpoint may reference older filing-history JSON files, but those are unnecessary for this company-level join. Full filing history would require following those additional files in a later analysis.

In [1]:
from pathlib import Path
import io, json, time
import requests
import pandas as pd
from bs4 import BeautifulSoup

RAW = Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('data/processed'); PROCESSED.mkdir(parents=True, exist_ok=True)
HEADERS = {'User-Agent': 'DSA405-coursework-nmbrockm/1.0 (student research; contact: nmbrockm@ncsu.edu)', 'Accept': 'application/json, text/html'}
N_COMPANIES = 5
_last_request = [0.0]

def fetch_cached(url, filename, *, session=None):
    """Read cached raw bytes or fetch once, rate-limit, validate and atomically cache."""
    path = RAW / filename
    if path.exists():
        payload = path.read_bytes()
        if not payload:
            raise ValueError(f'Empty cached response: {path}')
        print('CACHE:', path)
        return payload
    elapsed = time.monotonic() - _last_request[0]
    if elapsed < 1.0:
        time.sleep(1.0 - elapsed)
    session = session or requests
    try:
        response = session.get(url, headers=HEADERS, timeout=(10, 30))
    except requests.RequestException as exc:
        raise RuntimeError(f"Could not retrieve {url}: {exc}") from exc
    finally:
        _last_request[0] = time.monotonic()
    if 'data.sec.gov' in url:
        actual = {k: v for k, v in response.headers.items() if any(t in k.lower() for t in ('ratelimit', 'rate-limit', 'retry-after'))}
        print('SEC API rate-limit response headers:', actual if actual else 'none supplied; using >=1 second/request')
        if 'Retry-After' in response.headers:
            raise RuntimeError(f'SEC requested a pause (Retry-After: {response.headers["Retry-After"]}); stop and retry later')
    if response.status_code != 200:
        raise RuntimeError(f'HTTP {response.status_code} for {url}; stopping, no cache written')
    payload = response.content
    if not payload:
        raise ValueError(f'Empty HTTP response for {url}')
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_bytes(payload)
    temp.replace(path)
    print('FETCHED:', path, len(payload), 'bytes')
    return payload


### Demonstrated fail-loudly test (no external request)
A fake server returns HTTP 503. The function must raise a visible exception and must **not** cache the error response.

In [2]:
class FakeResponse:
    status_code = 503
    content = b'service unavailable'
    headers = {}
class FakeSession:
    def get(self, *args, **kwargs):
        return FakeResponse()

failure_path = RAW / '_deliberate_failure_test.html'
if failure_path.exists(): failure_path.unlink()
try:
    fetch_cached('https://example.org/deliberate-test', failure_path.name, session=FakeSession())
    raise AssertionError('FAIL: HTTP error was not caught')
except RuntimeError as exc:
    assert 'HTTP 503' in str(exc)
    assert not failure_path.exists()
    print('PASS: failed loudly and did not cache the error:', exc)


PASS: failed loudly and did not cache the error: HTTP 503 for https://example.org/deliberate-test; stopping, no cache written


### Retrieve and parse Wikipedia
Do not assume the first HTML table is the right one: find the constituent table by its stable `id`. Raise an exception if the selector matches nothing or expected columns are missing.

In [3]:

import io
import pandas as pd
from bs4 import BeautifulSoup

wiki_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

wiki_bytes = fetch_cached(
    wiki_url,
    "wikipedia_sp500.html"
)

soup = BeautifulSoup(wiki_bytes, "html.parser")

table = soup.select_one("table#constituents")

if table is None:
    raise ValueError("Wikipedia S&P 500 table not found")

wiki_tables = pd.read_html(io.StringIO(str(table)))

if len(wiki_tables) != 1:
    raise ValueError("Unexpected number of Wikipedia tables")

constituents = wiki_tables[0].copy()

required_columns = {"Symbol", "Security", "CIK"}

missing_columns = required_columns - set(constituents.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

constituents["CIK"] = pd.to_numeric(
    constituents["CIK"],
    errors="raise"
).astype("int64")

assert constituents["CIK"].notna().all()

print("Total S&P 500 securities:", len(constituents))

duplicates = constituents[
    constituents.duplicated(
        subset="CIK",
        keep=False
    )
].sort_values("CIK")

print("Rows sharing a CIK:", len(duplicates))

if not duplicates.empty:
    display(duplicates[["Symbol", "Security", "CIK"]])

companies = (
    constituents
    .sort_values("Symbol")
    .drop_duplicates(subset="CIK", keep="first")
    .reset_index(drop=True)
)

assert companies["CIK"].is_unique

print("Unique companies:", len(companies))

N_COMPANIES = 5

left = (
    companies[["CIK", "Symbol", "Security"]]
    .head(N_COMPANIES)
    .reset_index(drop=True)
)

assert len(left) == N_COMPANIES
assert left["CIK"].is_unique

print("Wikipedia acquisition completed successfully!")
print("Selected companies:", len(left))

display(left)



FETCHED: data/raw/wikipedia_sp500.html 571857 bytes
Total S&P 500 securities: 503
Rows sharing a CIK: 6


,Symbol,Security,CIK
330,NWS,News Corp (Class B),1564708
329,NWSA,News Corp (Class A),1564708
19,GOOGL,Alphabet Inc. (Class A),1652044
20,GOOG,Alphabet Inc. (Class C),1652044
204,FOX,Fox Corporation (Class B),1754301
203,FOXA,Fox Corporation (Class A),1754301


Unique companies: 500
Wikipedia acquisition completed successfully!
Selected companies: 5


,CIK,Symbol,Security
0,1090872,A,Agilent Technologies
1,320193,AAPL,Apple Inc.
2,1551152,ABBV,AbbVie
3,1559720,ABNB,Airbnb
4,1800,ABT,Abbott Laboratories


### Retrieve SEC submissions
SEC company-submission URLs are determined by CIK, with zero-padding to ten digits. The `filings.recent` arrays contain recent filings; older filings may be linked under `filings.files` (not needed for this company-level demonstration). We deliberately do not follow filing-document URLs here.

In [4]:

sec_rows = []

for cik in left["CIK"]:
    url = (
        "https://data.sec.gov/submissions/"
        f"CIK{int(cik):010d}.json"
    )

    raw = fetch_cached(
        url,
        f"sec_CIK{int(cik):010d}.json"
    )

    try:
        record = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid SEC JSON for CIK {cik}"
        ) from exc

    if not all(
        key in record for key in ("cik", "name", "filings")
    ):
        raise ValueError(
            f"Missing SEC fields for CIK {cik}"
        )

    if int(record["cik"]) != int(cik):
        raise ValueError(
            f"SEC CIK mismatch for {cik}"
        )

    recent = record["filings"].get("recent", {})
    forms = recent.get("form")

    if not isinstance(forms, list):
        raise ValueError(
            f"Missing filing forms for CIK {cik}"
        )

    sec_rows.append({
        "CIK": int(record["cik"]),
        "sec_company_name": record["name"],
        "recent_10k_count": forms.count("10-K")
    })

right = pd.DataFrame(sec_rows)

assert len(right) == len(left)
assert right["CIK"].is_unique

print("SEC acquisition completed successfully!")
print("SEC records collected:", len(right))

display(right)



SEC API rate-limit response headers: none supplied; using >=1 second/request
FETCHED: data/raw/sec_CIK0001090872.json 158120 bytes
SEC API rate-limit response headers: none supplied; using >=1 second/request
FETCHED: data/raw/sec_CIK0000320193.json 164219 bytes
SEC API rate-limit response headers: none supplied; using >=1 second/request
FETCHED: data/raw/sec_CIK0001551152.json 157038 bytes
SEC API rate-limit response headers: none supplied; using >=1 second/request
FETCHED: data/raw/sec_CIK0001559720.json 151527 bytes
SEC API rate-limit response headers: none supplied; using >=1 second/request
FETCHED: data/raw/sec_CIK0000001800.json 155913 bytes
SEC acquisition completed successfully!
SEC records collected: 5


,CIK,sec_company_name,recent_10k_count
0,1090872,"AGILENT TECHNOLOGIES, INC.",10
1,320193,Apple Inc.,12
2,1551152,AbbVie Inc.,9
3,1559720,"Airbnb, Inc.",3
4,1800,ABBOTT LABORATORIES,8


## 2. Ethics and legality in practice
**Consistency with A8:** The original plan named SEC EDGAR, Wikipedia and a conditional Stooq source. This notebook follows the public SEC API and Wikipedia portion, with honest identification, throttling and raw-response caching. No login, paywall, personal data extraction, access-control circumvention or prohibited path is used.

**Drift from A8:** A8 proposed analyzing thousands of filings and potentially stock prices; this notebook instead integrates a small reproducible company-level sample, because complete Item 1A extraction and the 12-month return calculation are not available from the attached A8 work. Stooq remains excluded because the A8 permission question is unresolved. The final research question **cannot yet be answered** with this join alone. Verify current site policies before scaling acquisition.

## 3. Join design — written BEFORE merge

| Question | Decision |
|---|---|
| Join key | `CIK` (numeric SEC company identifier) |
| Left grain | One row per selected S&P 500 constituent/company |
| Right grain | One row per SEC company submission record |
| Expected relationship | One-to-one (`one_to_one`) |
| Expected output count | Exactly `len(left)` = **5** with default sample size |
| Join type | Left: retain every selected constituent and reveal missing SEC matches |
| Unmatched rows | Kept with missing SEC fields and `_merge == left_only`; assertions flag them |

**Why not ticker or company name?** Names can differ between sources; tickers may be formatted differently or change over time. SEC CIK is the more stable common identifier.

**Which failure is guarded against?** `validate='one_to_one'` and row-count checks protect against **fan-out**. A left join avoids silently deleting left rows (the **inner-join wreck**), while the match-rate assertion catches missing SEC records. This is a current-constituents sample and should not be mistaken for historical S&P 500 membership.

**Duplicate CIKs:** Wikipedia returned 503 securities but 500 unique CIKs. Alphabet, News Corp, and Fox each have two share classes. I kept the raw table as-is and made a separate company-level table with one ticker per CIK before merging. This works for checking company-level SEC metadata, but I would need to revisit the share-class choice before analyzing stock returns.

In [5]:

expected_n = len(left)

assert expected_n == N_COMPANIES
assert left["CIK"].is_unique
assert right["CIK"].is_unique

joined = left.merge(
    right,
    on="CIK",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Rows before merge:", len(left))
print("Rows after merge:", len(joined))
print("Matched companies:", joined["_merge"].eq("both").sum())

display(joined)



Rows before merge: 5
Rows after merge: 5
Matched companies: 5


,CIK,Symbol,Security,sec_company_name,recent_10k_count,_merge
0,1090872,A,Agilent Technologies,"AGILENT TECHNOLOGIES, INC.",10,both
1,320193,AAPL,Apple Inc.,Apple Inc.,12,both
2,1551152,ABBV,AbbVie,AbbVie Inc.,9,both
3,1559720,ABNB,Airbnb,"Airbnb, Inc.",3,both
4,1800,ABT,Abbott Laboratories,ABBOTT LABORATORIES,8,both


## 4. Verification suite
Each assertion checks an assumption about the join. I added a short comment above each check to explain what could go wrong.

In [6]:
# Makes sure the merge did not add extra rows or drop any companies.
assert len(joined) == expected_n, f"Expected {expected_n} rows, got {len(joined)}"

# Makes sure each company only shows up once after the merge.
assert joined["CIK"].is_unique, "Duplicate CIK found after merging"

# Makes sure every selected company matched an SEC record.
assert joined["_merge"].eq("both").all(), "Some companies did not match the SEC data"

# Makes sure the 10-K counts are present and cannot be negative.
assert joined["recent_10k_count"].notna().all() and joined["recent_10k_count"].ge(0).all(), "Missing or invalid 10-K counts"

# Makes sure the merge kept the same company identifiers.
assert set(joined["CIK"]) == set(left["CIK"]), "The company identifiers changed"

print("PASS: All 5 commented verification checks passed")
print("Rows:", len(joined))
print("Matched:", int(joined["_merge"].eq("both").sum()))
print("Match rate:", f"{joined['_merge'].eq('both').mean():.1%}")
joined.drop(columns="_merge").to_csv(PROCESSED / "p3_company_join.csv", index=False)
print("Saved:", PROCESSED / "p3_company_join.csv")

PASS: All 5 commented verification checks passed
Rows: 5
Matched: 5
Match rate: 100.0%
Saved: data/processed/p3_company_join.csv


### Demonstrate cardinality protection with a deliberately bad right table
The extra SEC row duplicates a CIK; pandas must reject it as a one-to-one merge. This is a separate controlled failure test and does not alter the real output.

In [7]:
bad_right = pd.concat([right, right.iloc[[0]]], ignore_index=True)
try:
    left.merge(bad_right, on='CIK', how='left', validate='one_to_one')
    raise AssertionError('FAIL: duplicate SEC CIK was not rejected')
except pd.errors.MergeError as exc:
    print('PASS: pandas caught deliberate fan-out risk:', exc)


PASS: pandas caught deliberate fan-out risk: Merge keys are not unique in right dataset; not a one-to-one merge


## 5. What surprised me / what I checked
One thing that surprised me was that the Wikipedia S&P 500 table had 503 securities but only 500 unique companies. I assumed each row would have its own CIK, but Alphabet, News Corp, and Fox have different stock classes that share the same CIK. My first uniqueness check actually failed because of this. I fixed it by making a separate company-level table without changing the original raw data.

I then used five companies to test the SEC join. All five matched, the row count stayed at five, and the match rate was 100%. The verification checks also passed. This helped me see why checking the data before and after merging is important.

## 6. Reproducibility and submission
In Colab, use **Runtime → Restart session and run all**. Make sure every cell finishes without an unexpected error. The notebook stores downloaded responses in `data/raw/` and saves the merged company table to `data/processed/p3_company_join.csv`. Colab storage is temporary unless the files are saved to Drive or downloaded. Keep the raw responses unchanged. Update `README.md` in GitHub and submit the notebook and repository link.

**Repository:** https://github.com/nmb1106-wq/dsa405-project

### Self-scored rubric (provisional)
| Criterion | Weight | Self-score (0–2) | Evidence / limitation |
|---|---:|---:|---|
| Acquisition | ×2 | 1 | Caching, rate limiting, API-header inspection, and a caught HTTP 503 failure; this is only a small company-level sample |
| Ethics & legality | ×2 | 2 | A8 alignment, access guardrails, and the change in scope are explained |
| Join design | ×2 | 2 | Key, grain, expected count, join type, duplicate CIK handling, and fan-out protection are explained |
| Verification | ×3 | 2 | Five commented checks and a deliberate invalid join test |
| Reproducibility | ×1 | 1 | Notebook is organized, but the revised version still needs a fresh-kernel run and the README must be checked |
| **Weighted total** | | | **17/20 (provisional)** |

**AI use:** I used AI to help organize the notebook and troubleshoot code. I checked the outputs myself and am responsible for verifying the results and following the source rules.

**Scope note:** This is a five-company integration of Wikipedia and SEC filing metadata. It does not yet extract Item 1A risk-factor text or calculate subsequent stock returns. Those steps would be needed to answer my original research question.